# Nepali Deep Learning Model (XLM-R + CNN)
This notebook demonstrates a deep learning pipeline for Nepali (Devanagari) text.

In [ ]:

!pip install torch transformers


## Imports and Tokenizer

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import XLMRobertaTokenizer, XLMRobertaModel


In [ ]:

tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

def tokenize_text(texts, max_len=128):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )


## Model Definition

In [ ]:

class NepaliCNNTransformer(nn.Module):
    def __init__(self, num_classes, filters=128, kernel_size=3):
        super().__init__()

        self.encoder = XLMRobertaModel.from_pretrained("xlm-roberta-base")
        hidden_size = self.encoder.config.hidden_size

        self.conv1d = nn.Conv1d(
            in_channels=hidden_size,
            out_channels=filters,
            kernel_size=kernel_size,
            padding=1
        )

        fusion_dim = hidden_size + filters
        self.fc1 = nn.Linear(fusion_dim, 64)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        token_embeddings = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()

        mean_pool = torch.sum(token_embeddings * mask, dim=1)
        mean_pool = mean_pool / torch.clamp(mask.sum(dim=1), min=1e-9)

        x = token_embeddings.permute(0, 2, 1)
        conv_out = F.relu(self.conv1d(x))
        global_max_pool = torch.max(conv_out, dim=2)[0]

        fused = torch.cat([mean_pool, global_max_pool], dim=1)

        x = F.relu(self.fc1(fused))
        x = self.dropout(x)
        logits = self.classifier(x)

        return logits


## Example Data and Forward Pass

In [ ]:

texts = [
    "नेपाल एक सुन्दर देश हो",
    "आज मौसम धेरै राम्रो छ"
]

labels = torch.tensor([0, 1])

inputs = tokenize_text(texts)

model = NepaliCNNTransformer(num_classes=2)

outputs = model(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"]
)

outputs


## Training Loop

In [ ]:

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

model.train()

for epoch in range(3):
    optimizer.zero_grad()

    logits = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
    )

    loss = criterion(logits, labels)
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1} | Loss: {loss.item():.4f}")
